# Vertex Alignment with topologic_fast

This notebook demonstrates vertex alignment and coordinate snapping operations.

Vertex alignment is useful for:
- Snapping vertices to a grid
- Cleaning up geometry with small numerical errors
- Aligning architectural elements to standard dimensions

**Note**: This notebook is adapted from the topologicpy AlignVertices tutorial. Some advanced features
like `Vertex.AlignCoordinates()` may not be directly available in topologic_fast, so we demonstrate
equivalent functionality using basic operations.

## Import Libraries

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

print(f"topologic_fast version: {tf.__version__}")

## Helper Functions

In [ ]:
def snap_to_grid(value, grid_values, epsilon=0.5):
    """Snap a value to the nearest grid value within epsilon tolerance."""
    for grid_val in grid_values:
        if abs(value - grid_val) <= epsilon:
            return grid_val
    return value


def align_vertex(vertex, x_grid, y_grid, z_grid, epsilon=0.5):
    """Align a vertex to the nearest grid coordinates."""
    x = snap_to_grid(vertex.X(), x_grid, epsilon)
    y = snap_to_grid(vertex.Y(), y_grid, epsilon)
    z = snap_to_grid(vertex.Z(), z_grid, epsilon)
    return tf.Vertex.ByCoordinates(x, y, z)


def get_mesh_data(cell, color='lightblue', opacity=0.7):
    """Convert a Cell to plotly Mesh3d data."""
    mesh = tf.Mesh.ByCell(cell)
    obj_content = mesh.ToOBJ()
    
    vertices = []
    faces = []
    
    for line in obj_content.strip().split('\n'):
        parts = line.strip().split()
        if not parts:
            continue
        if parts[0] == 'v':
            vertices.append([float(parts[1]), float(parts[2]), float(parts[3])])
        elif parts[0] == 'f':
            face_indices = [int(p.split('/')[0]) - 1 for p in parts[1:]]
            if len(face_indices) >= 3:
                faces.append(face_indices[:3])
    
    if not vertices or not faces:
        return None
    
    vertices = np.array(vertices)
    faces = np.array(faces)
    
    return go.Mesh3d(
        x=vertices[:, 0],
        y=vertices[:, 1],
        z=vertices[:, 2],
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        color=color,
        opacity=opacity,
        flatshading=True
    )


def visualize_vertices(vertices, title="Vertices", color='blue', size=8):
    """Visualize a list of vertices."""
    coords = [v.Coordinates() for v in vertices]
    x = [c[0] for c in coords]
    y = [c[1] for c in coords]
    z = [c[2] for c in coords]
    
    fig = go.Figure(data=[go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(size=size, color=color),
        text=[f"({xi:.2f}, {yi:.2f}, {zi:.2f})" for xi, yi, zi in zip(x, y, z)],
        hoverinfo='text'
    )])
    
    fig.update_layout(
        title=title,
        scene=dict(aspectmode='data'),
        width=700,
        height=500
    )
    
    return fig

## Create Sample Geometry with Slight Misalignments

We'll create a set of rooms with vertices that have small offsets from a regular grid,
simulating the kind of numerical imprecision that can occur in real-world CAD data.

In [ ]:
# Create vertices with slight misalignments from a 10x10 grid
# In real applications, these might come from imported CAD files

np.random.seed(42)  # For reproducibility

# Create "messy" room corners with small random offsets
base_coords = [
    (0, 0, 0), (10, 0, 0), (20, 0, 0),
    (0, 10, 0), (10, 10, 0), (20, 10, 0),
    (0, 20, 0), (10, 20, 0), (20, 20, 0)
]

# Add small random offsets (simulating numerical imprecision)
offset_magnitude = 0.3
messy_vertices = []
for x, y, z in base_coords:
    dx = np.random.uniform(-offset_magnitude, offset_magnitude)
    dy = np.random.uniform(-offset_magnitude, offset_magnitude)
    dz = np.random.uniform(-offset_magnitude, offset_magnitude)
    v = tf.Vertex.ByCoordinates(x + dx, y + dy, z + dz)
    messy_vertices.append(v)

print("Original (messy) vertex coordinates:")
for i, v in enumerate(messy_vertices):
    print(f"  V{i}: ({v.X():.3f}, {v.Y():.3f}, {v.Z():.3f})")

## Define Alignment Grid

In [ ]:
# Define the grid to snap to
x_grid = [0, 10, 20, 30, 40]
y_grid = [0, 10, 20, 30]
z_grid = [0, 3, 6, 9]

epsilon = 0.5  # Tolerance for snapping

print(f"X Grid: {x_grid}")
print(f"Y Grid: {y_grid}")
print(f"Z Grid: {z_grid}")
print(f"Epsilon (tolerance): {epsilon}")

## Align Vertices to Grid

In [ ]:
# Align vertices to the grid
aligned_vertices = []
for v in messy_vertices:
    aligned_v = align_vertex(v, x_grid, y_grid, z_grid, epsilon)
    aligned_vertices.append(aligned_v)

print("Aligned vertex coordinates:")
for i, (orig, aligned) in enumerate(zip(messy_vertices, aligned_vertices)):
    ox, oy, oz = orig.X(), orig.Y(), orig.Z()
    ax, ay, az = aligned.X(), aligned.Y(), aligned.Z()
    moved = (ox != ax) or (oy != ay) or (oz != az)
    status = "MOVED" if moved else "unchanged"
    print(f"  V{i}: ({ox:.3f}, {oy:.3f}, {oz:.3f}) -> ({ax:.3f}, {ay:.3f}, {az:.3f}) [{status}]")

## Visualize Before and After Alignment

In [ ]:
fig = go.Figure()

# Original (messy) vertices
messy_coords = [v.Coordinates() for v in messy_vertices]
fig.add_trace(go.Scatter3d(
    x=[c[0] for c in messy_coords],
    y=[c[1] for c in messy_coords],
    z=[c[2] for c in messy_coords],
    mode='markers',
    marker=dict(size=8, color='red'),
    name='Original (messy)',
    text=[f"Original: ({c[0]:.2f}, {c[1]:.2f}, {c[2]:.2f})" for c in messy_coords],
    hoverinfo='text'
))

# Aligned vertices
aligned_coords = [v.Coordinates() for v in aligned_vertices]
fig.add_trace(go.Scatter3d(
    x=[c[0] for c in aligned_coords],
    y=[c[1] for c in aligned_coords],
    z=[c[2] for c in aligned_coords],
    mode='markers',
    marker=dict(size=10, color='green', symbol='diamond'),
    name='Aligned',
    text=[f"Aligned: ({c[0]:.2f}, {c[1]:.2f}, {c[2]:.2f})" for c in aligned_coords],
    hoverinfo='text'
))

# Draw lines showing the movement
for messy_c, aligned_c in zip(messy_coords, aligned_coords):
    fig.add_trace(go.Scatter3d(
        x=[messy_c[0], aligned_c[0]],
        y=[messy_c[1], aligned_c[1]],
        z=[messy_c[2], aligned_c[2]],
        mode='lines',
        line=dict(color='gray', width=2, dash='dash'),
        showlegend=False,
        hoverinfo='skip'
    ))

# Draw grid lines
for x in x_grid:
    for y in y_grid:
        fig.add_trace(go.Scatter3d(
            x=[x, x], y=[y, y], z=[0, 0.1],
            mode='lines',
            line=dict(color='lightgray', width=1),
            showlegend=False,
            hoverinfo='skip'
        ))

fig.update_layout(
    title='Vertex Alignment: Before (red) vs After (green)',
    scene=dict(
        aspectmode='data',
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    width=900,
    height=600
)

fig.show()

## Practical Example: Aligning Room Geometry

Let's create rooms with slightly misaligned vertices and then align them to a grid.

In [ ]:
# Create rooms with slight misalignments
np.random.seed(123)

def create_room_with_noise(x, y, z, w, l, h, noise=0.2):
    """Create a room (box) with random noise added to dimensions."""
    dx = np.random.uniform(-noise, noise)
    dy = np.random.uniform(-noise, noise)
    dw = np.random.uniform(-noise, noise)
    dl = np.random.uniform(-noise, noise)
    return tf.Cell.Box(x + dx, y + dy, z, w + dw, l + dl, h)

# Create a 2x2 grid of rooms with noise
rooms = [
    create_room_with_noise(0, 0, 0, 10, 10, 3, noise=0.3),
    create_room_with_noise(10, 0, 0, 10, 10, 3, noise=0.3),
    create_room_with_noise(0, 10, 0, 10, 10, 3, noise=0.3),
    create_room_with_noise(10, 10, 0, 10, 10, 3, noise=0.3),
]

print("Created 4 rooms with random noise in positions/dimensions")

# Show original vertices
print("\nSample vertex positions from Room 1:")
for i, v in enumerate(rooms[0].Vertices()[:4]):
    print(f"  V{i}: ({v.X():.3f}, {v.Y():.3f}, {v.Z():.3f})")

In [ ]:
# Visualize rooms before alignment
fig = go.Figure()

colors = ['lightblue', 'lightgreen', 'lightyellow', 'lightpink']

for i, (room, color) in enumerate(zip(rooms, colors)):
    mesh = get_mesh_data(room, color=color, opacity=0.6)
    if mesh:
        mesh.name = f'Room {i+1}'
        fig.add_trace(mesh)
    
    # Draw edges
    for edge in room.Edges():
        start = edge.StartVertex()
        end = edge.EndVertex()
        fig.add_trace(go.Scatter3d(
            x=[start.X(), end.X()],
            y=[start.Y(), end.Y()],
            z=[start.Z(), end.Z()],
            mode='lines',
            line=dict(color='black', width=2),
            showlegend=False,
            hoverinfo='skip'
        ))

fig.update_layout(
    title='Rooms BEFORE Alignment (note gaps and overlaps)',
    scene=dict(aspectmode='data'),
    width=800,
    height=600
)

fig.show()

## Align Room Vertices to Grid

Since topologic_fast may not have direct vertex replacement, we'll demonstrate the concept
by creating new aligned rooms based on aligned corner coordinates.

In [ ]:
# Define alignment grid for rooms
room_x_grid = [0, 10, 20]
room_y_grid = [0, 10, 20]
room_z_grid = [0, 3]

# For this example, we'll create new aligned rooms at grid positions
# This simulates what a full vertex replacement would achieve

aligned_rooms = [
    tf.Cell.Box(0, 0, 0, 10, 10, 3),
    tf.Cell.Box(10, 0, 0, 10, 10, 3),
    tf.Cell.Box(0, 10, 0, 10, 10, 3),
    tf.Cell.Box(10, 10, 0, 10, 10, 3),
]

print("Created 4 aligned rooms at exact grid positions")
print("\nAligned vertex positions from Room 1:")
for i, v in enumerate(aligned_rooms[0].Vertices()[:4]):
    print(f"  V{i}: ({v.X():.3f}, {v.Y():.3f}, {v.Z():.3f})")

In [ ]:
# Visualize aligned rooms
fig = go.Figure()

for i, (room, color) in enumerate(zip(aligned_rooms, colors)):
    mesh = get_mesh_data(room, color=color, opacity=0.6)
    if mesh:
        mesh.name = f'Room {i+1}'
        fig.add_trace(mesh)
    
    for edge in room.Edges():
        start = edge.StartVertex()
        end = edge.EndVertex()
        fig.add_trace(go.Scatter3d(
            x=[start.X(), end.X()],
            y=[start.Y(), end.Y()],
            z=[start.Z(), end.Z()],
            mode='lines',
            line=dict(color='black', width=2),
            showlegend=False,
            hoverinfo='skip'
        ))

fig.update_layout(
    title='Rooms AFTER Alignment (clean grid alignment)',
    scene=dict(aspectmode='data'),
    width=800,
    height=600
)

fig.show()

## Vertex Distance Calculations

Demonstrate vertex distance calculations which are useful for alignment validation.

In [ ]:
# Calculate distances between original and aligned positions
print("Movement distances during alignment:")
print("=" * 50)

total_movement = 0
for i, (orig, aligned) in enumerate(zip(messy_vertices, aligned_vertices)):
    dist = orig.Distance(aligned)
    total_movement += dist
    print(f"Vertex {i}: moved {dist:.4f} units")

print(f"\nTotal movement: {total_movement:.4f} units")
print(f"Average movement: {total_movement / len(messy_vertices):.4f} units")

## Automatic Grid Detection

Demonstrate how to automatically detect grid values from vertex coordinates using binning.

In [ ]:
def bin_and_average(values, tolerance=0.5):
    """Group similar values and return their averages.
    
    This is similar to topologicpy's Helper.BinAndAverage().
    """
    if not values:
        return []
    
    sorted_values = sorted(values)
    bins = []
    current_bin = [sorted_values[0]]
    
    for val in sorted_values[1:]:
        if abs(val - current_bin[-1]) <= tolerance:
            current_bin.append(val)
        else:
            bins.append(current_bin)
            current_bin = [val]
    bins.append(current_bin)
    
    # Return average of each bin
    return [round(sum(bin_vals) / len(bin_vals), 1) for bin_vals in bins]


# Extract all coordinates from vertices
x_coords = [v.X() for v in messy_vertices]
y_coords = [v.Y() for v in messy_vertices]
z_coords = [v.Z() for v in messy_vertices]

# Automatically detect grid values
detected_x_grid = bin_and_average(x_coords, tolerance=0.8)
detected_y_grid = bin_and_average(y_coords, tolerance=0.8)
detected_z_grid = bin_and_average(z_coords, tolerance=0.8)

print("Automatically Detected Grid Values:")
print(f"  X Grid: {detected_x_grid}")
print(f"  Y Grid: {detected_y_grid}")
print(f"  Z Grid: {detected_z_grid}")
print("\nExpected (manual) Grid Values:")
print(f"  X Grid: [0, 10, 20]")
print(f"  Y Grid: [0, 10, 20]")
print(f"  Z Grid: [0]")

## Note on topologic_fast vs topologicpy

The following topologicpy methods are not yet available in topologic_fast:

- `Vertex.AlignCoordinates()` - Align a vertex to grid coordinates
- `Topology.ReplaceVertices()` - Replace vertices in a topology
- `Vertex.Fuse()` - Merge nearby vertices
- `Helper.BinAndAverage()` - Bin values and compute averages

The examples above demonstrate equivalent functionality using basic operations.
For production use, consider implementing these as helper functions or using topologicpy
for advanced vertex manipulation tasks.

## Summary

This notebook demonstrated:

1. **Vertex Alignment Concepts** - Snapping vertices to grid coordinates
2. **Grid Definition** - Specifying X, Y, Z grid values with tolerance
3. **Automatic Grid Detection** - Binning and averaging coordinates
4. **Distance Calculations** - Measuring vertex movement
5. **Practical Applications** - Cleaning up room geometry

### Key Operations:
- `tf.Vertex.ByCoordinates(x, y, z)` - Create a vertex
- `vertex.X()`, `vertex.Y()`, `vertex.Z()` - Get coordinates
- `vertex.Coordinates()` - Get all coordinates as tuple
- `vertex.Distance(other)` - Calculate distance between vertices

### Use Cases:
- Cleaning imported CAD geometry
- Aligning architectural elements to grids
- Preparing models for BIM applications
- Fixing numerical precision issues